# 3장 Candidate B 대상 배포 확인

## 이번 질문

플랫폼이 이미 등록한 Argo Application을 Candidate B overlay로 맞춘 뒤, 대상 `AIQA_RISK_API_URL`의 `/v1/model`이 `candidate-b-c712a8e52344`인지 확인합니다. Application 생성, KServe 설치, GHCR pull secret은 이 노트북의 범위가 아닙니다. 로컬 Compose 주소와 대상 URL을 섞지 않습니다. 대상 URL이 없거나 200이 아니면 identity를 만들지 않고 `API_NOT_RUNNING`과 `operational_deployment_scope=target_pending`으로 멈춥니다. 이 확인이 통과해도 공식 평가나 sealed test를 다시 실행한 것이 아닙니다.

## 먼저 예상

대상 URL이 없을 때 어떤 운영 scope와 실행 결과를 기록할지 한 문장으로 적습니다. `/v1/model`의 `version`이 `candidate-b-c712a8e52344`와 다를 때 모델 승인을 바꿀지, 아니면 `result=BLOCKED`만 남길지도 예상합니다.

## 실행과 관측

In [ ]:
from pathlib import Path

import pandas as pd

# 1. 저장소 루트를 찾아 이후 셀이 같은 경로를 쓰게 합니다.
ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir()
)
pd.Series({"repository_root": ROOT.name})

### 1. 승인된 Candidate B overlay identity를 읽는다

배포 선언이 고르는 모델 경로는 `candidate-b-c712a8e52344`입니다. Candidate A는 이 overlay에 없어야 합니다. 이 읽기는 정적 선언이며 대상 동기화 PASS가 아닙니다.

In [ ]:
# 1. Candidate B overlay만 읽고 승인된 digest를 표로 남깁니다.
# 2. Candidate A 문자열이 있으면 대상 확인을 진행하지 않습니다.
overlay_path = ROOT / "deploy/kubernetes/overlays/candidate-b/kustomization.yaml"
overlay_text = overlay_path.read_text(encoding="utf-8")
expected_version = "candidate-b-c712a8e52344"
row = {
    "overlay": str(overlay_path.relative_to(ROOT)),
    "expected_version": expected_version,
    "contains_expected_digest": expected_version in overlay_text,
    "contains_candidate_a": "candidate-a" in overlay_text,
}
if row["contains_candidate_a"] or not row["contains_expected_digest"]:
    row["result"] = "BLOCKED"
    row["reason"] = "Candidate B overlay identity가 선언과 다릅니다."
else:
    row["result"] = "static"
    row["reason"] = "정적 overlay는 승인된 Candidate B만 선택합니다."
pd.DataFrame([row])

### 2. 문서화된 동기화 스크립트를 호출한다

클러스터 변경은 `scripts/sync_student_release.py`만 사용합니다. 스크립트는 이미 등록된 Application만 Candidate B overlay로 바꾸고, 없으면 Application을 만들지 않습니다. 이름이 없으면 `result=BLOCKED`로 멈춥니다.

In [ ]:
import os
import subprocess

# 1. 클러스터 변경은 문서화된 동기화 스크립트만 호출합니다.
# 2. Application 이름이 없으면 만들지 않고 멈춥니다.
application_name = os.getenv("AIQA_ARGOCD_APPLICATION_NAME")
sync_command = [
    "uv",
    "run",
    "python",
    "scripts/sync_student_release.py",
]
row = {
    "command": "uv run python scripts/sync_student_release.py --application-name <already-registered>",
    "result": "BLOCKED",
    "reason": "AIQA_ARGOCD_APPLICATION_NAME이 없어 기존 Application만 동기화할 수 있습니다.",
}
if application_name:
    sync_command.extend(["--application-name", application_name])
    completed = subprocess.run(
        sync_command,
        cwd=ROOT,
        capture_output=True,
        text=True,
    )
    detail = (completed.stdout or completed.stderr).strip() or "동기화가 막혔습니다."
    row = {
        "command": " ".join(sync_command),
        "result": "synced" if completed.returncode == 0 else "BLOCKED",
        "reason": detail,
    }
pd.DataFrame([row])

### 3. 대상 `/v1/model`을 Candidate B digest와 대조한다

대상 주소는 `AIQA_RISK_API_URL`만 사용합니다. URL이 없거나 응답이 200이 아니면 `API_NOT_RUNNING`과 `operational_deployment_scope=target_pending` 표에서 멈추고 version을 만들지 않습니다.

In [ ]:
import os

import pandas as pd
import requests

# 1. 대상 URL은 AIQA_RISK_API_URL만 사용합니다.
# 2. URL이 없으면 identity를 만들지 않고 멈춥니다.
# 3. 200이 아니면 target_pending으로 멈춥니다.
expected_version = "candidate-b-c712a8e52344"
api_url = os.getenv("AIQA_RISK_API_URL")
row = {
    "status": "API_NOT_RUNNING",
    "operational_deployment_scope": "target_pending",
    "result": "BLOCKED",
    "reason": "AIQA_RISK_API_URL이 없습니다.",
    "profile": pd.NA,
    "version": pd.NA,
    "matches_candidate_b": False,
}
if api_url:
    try:
        response = requests.get(f"{api_url.rstrip('/')}/v1/model", timeout=5)
        if response.status_code != 200:
            row = {
                "status": "API_NOT_RUNNING",
                "operational_deployment_scope": "target_pending",
                "result": "BLOCKED",
                "reason": f"HTTP {response.status_code}",
                "profile": pd.NA,
                "version": pd.NA,
                "matches_candidate_b": False,
            }
        else:
            body = response.json()
            version = body.get("version")
            matched = version == expected_version
            row = {
                "status": "ok" if matched else "mismatch",
                "operational_deployment_scope": "target" if matched else "target_pending",
                "result": "confirmed" if matched else "BLOCKED",
                "reason": (
                    "Candidate B digest가 대상 /v1/model과 같습니다."
                    if matched
                    else "대상 version이 candidate-b-c712a8e52344가 아닙니다."
                ),
                "profile": body.get("profile", pd.NA),
                "version": version if version is not None else pd.NA,
                "matches_candidate_b": matched,
            }
    except (OSError, ValueError, requests.RequestException) as error:
        row = {
            "status": "API_NOT_RUNNING",
            "operational_deployment_scope": "target_pending",
            "result": "BLOCKED",
            "reason": str(error),
            "profile": pd.NA,
            "version": pd.NA,
            "matches_candidate_b": False,
        }
pd.DataFrame([row])

## 해석과 판단 기록

확인한 overlay, 동기화 스크립트 결과, 대상 `/v1/model`만 근거로 상태를 고릅니다. 빠진 URL이나 비-200 응답을 성공으로 바꾸지 않습니다. 대상 digest가 맞아도 sealed test를 다시 평가했다고 쓰지 않습니다.

In [ ]:
# 1. 대상 identity가 없을 때 만든 값을 판단 기록에 넣지 않습니다.
# 2. 성공이어도 공식 평가 결과를 바꾸지 않습니다.
judgment = {
    "expected_version": "candidate-b-c712a8e52344",
    "observed_version": row.get("version", pd.NA),
    "operational_deployment_scope": row.get(
        "operational_deployment_scope", "target_pending"
    ),
    "result": row.get("result", "BLOCKED"),
    "reason": row.get("reason", "대상 /v1/model을 확인하지 못했습니다."),
    "sealed_test_reevaluated": False,
}
pd.DataFrame([judgment])

## 다음 확인

5장에서 Candidate B 모델 승인과 운영 환경 확인 상태를 한 기록에서 분리합니다. 대상 URL이나 동기화 결과가 없으면 `target_pending`을 유지합니다. Grafana 운영 기록은 같은 model identity와 시간 범위가 준비된 뒤에만 연결합니다.